# dyadmocap-temporal-mapper: one dyad

This notebook follows **one dyad** through the analysis: loading, skeleton and animation, joint angles, velocity, behavioural states and the Temporal Mapper graph. It ends with a table of the graph's main **graph and cycle metrics** for the two conversations. It does no statistics.

It uses **simulated** data, because the study's recordings are not public. The code is the same code that analysed the study data; only the input files differ. The numbers describe the simulation only; they are **not findings** about real conversations.

To follow another dyad, change `EXAMPLE` in the data cell below. The full analysis of all dyads, with statistics, is in `dyadmocap_walkthrough.ipynb`.

Temporal Mapper is an external dependency ([tmapper-py](https://github.com/Multiscale-Complex-Systems-Lab/tmapper-py)), installed by `environment.yml`.

## 0. Setup

Run this notebook from the `python/notebooks` folder, in the `dyadmocap-tm` environment (see the README). The first cell makes the package importable and checks that Temporal Mapper is installed.

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Image, Video

PYTHON_DIR = Path.cwd().parent            # .../python
REPO = PYTHON_DIR.parent
sys.path.insert(0, str(PYTHON_DIR))

import tmapper                              # external dependency
from dyadmocap import dymo_settings as S
print("Temporal Mapper (tmapper-py) found:", Path(tmapper.__file__).parent)
print("Dataset index :", S.DATA_INDEX)
print("Results folder:", S.PY_RESULTS_DIR)

def show(path, width=750):
    # show a saved PNG figure in the notebook
    display(Image(filename=str(path), width=width))

### The data

`data/simulated/simulate_dyad.py` writes 15 simulated dyads in the file layout of the study's motion-capture export (one tab-delimited file per person and conversation, 42 markers). Two dyads have a conversation that is too short **on purpose**, so the inclusion rule (below) keeps **13**. The files are written once (1–3 minutes, about 0.2 GB) with fixed seeds, so every run gives identical files.

To analyse your own data instead, copy `dymo_config.example.ini` to `dymo_config.ini` and point `data_index` at an index file of your recordings (the format is described in that file).

In [ ]:
if not Path(S.DATA_INDEX).is_file():
    print("Writing the simulated dyads (once)...")
    subprocess.run([sys.executable, str(REPO / "data" / "simulated" / "simulate_dyad.py")], check=True)

index = S.load_index()
dyads = index["dyads"]
n_short = sum(d.get("excluded_by_design", False) for d in dyads.values())
print("%d dyads in the index; %d too short by design" % (len(dyads), n_short))
EXAMPLE = "simdyad01"                       # the example dyad of Part A
TAG = " (simulated data)"


## A1. Loading, trimming and the analysis window

For each dyad and conversation (GTK = casual chat, MAN = conflict), the pipeline:

1. drops the first 25 s (calibration) and the last 5 s of each file;
2. finds a common, centred window of equal length in both conversations (3 s margin at each end);
3. fills short gaps (missing marker values) by linear interpolation;
4. **includes** a dyad only if every recording keeps at least 180 s;
5. analyses the first 180 s of each recording, so every recording has the same number of samples.

`load_group` does this for all dyads of one conversation.

In [ ]:
from dyadmocap.dymo_loading import find_times, load_group

times = find_times()
data, fs = {}, None
for group in S.GROUPS:
    data[group], _, fs, n_keep = load_group(group, times)
    print("%s: %d dyads included, %d samples per recording (%.0f s at %.0f Hz)"
          % (group, len(data[group]), n_keep, n_keep / fs, fs))
print("Excluded:", sorted(set(dyads) - set(data["GTK"])))

## A2. The skeleton, and an animation

The markers are placed in the room as in the MATLAB scripts (X = x, Y = −z, Z = y), with the seat marker replaced by the midpoint of the two hip markers (yellow). Person 1 is blue, Person 2 red. The animation is a check that the files are read and placed correctly: two seated people facing each other, moving.

In [ ]:
from dyadmocap import dymo_figures as F
from dyadmocap.dymo_animation import animate_dyad

WALK = Path(S.PY_RESULTS_DIR) / "walkthrough"
pair = data["GTK"][EXAMPLE]
F.skeleton_frame1(pair, "GTK " + EXAMPLE + " Skeleton View - Frame 1" + TAG, WALK, "Skeleton_Frame1")
show(WALK / "Skeleton_Frame1.png", 600)

video = animate_dyad(pair, WALK / "Skeleton_Animation", fs, seconds=10,
                     title="GTK " + EXAMPLE + TAG)
display(Video(video, embed=True, width=600) if video.endswith(".mp4") else Image(filename=video, width=600))

## A3. Joint angles and velocity

Eleven joint angles per person (shoulders, elbows, wrists, knees, ankles, head), each between 0° and 180°, from vectors between markers. The velocity is the change of each angle per second, for both people together (22 columns).

In [ ]:
from dyadmocap.dymo_angles import joint_angles, velocity

ang, vel = {}, {}
for group in S.GROUPS:
    p = data[group][EXAMPLE]
    a1, _ = joint_angles(p[0])
    a2, _ = joint_angles(p[1])
    ang[group], vel[group] = (a1, a2), velocity(a1, a2, fs)
    print("%s: angles %s per person, velocity %s" % (group, a1.shape, vel[group].shape))

F.angle_vector_skeletons(pair, "GTK " + EXAMPLE + TAG, WALK, "GTK")
F.angle_time_series(list(ang["GTK"]), [pair[0][:, 0], pair[1][:, 0]], "GTK " + EXAMPLE + TAG, WALK, "GTK")
show(WALK / "Elbow" / "GTK_Elbow_Angles_Skeleton.png", 600)
show(WALK / "All_Angles_Overview" / "GTK_All_Angles_Overview.png")

## A4. Behavioural states

Each person's velocities are median-normalised and combined into one activity score per moment. A person is **Active** when the score is above their own median, so each person is Active half of the time. Together the two people give four states per moment:

| state | colour |
|---|---|
| Both Inactive | black |
| Both Active | green |
| Person 1 Active | blue |
| Person 2 Active | red |

Because the data are simulated, we know the true (hidden) state at every moment and can check the rule against it. Agreement is high when a person really moves about half of the time and lower when they move much more or less (the rule still calls exactly half Active): this is the rule's built-in assumption.

In [ ]:
from dyadmocap.dymo_categories import normalize_median, categorize

N_ANG = 11
states = {}
for group in S.GROUPS:
    v = vel[group]
    r1, _ = normalize_median(v[:, :N_ANG])
    r2, _ = normalize_median(v[:, N_ANG:])
    states[group] = categorize(r1, r2)["colorvar"]
    share = [np.mean(states[group] == k) for k in (1, 2, 3, 4)]
    line = "%s: share of time in the 4 states = %s" % (group, np.round(share, 2))
    hfile = dyads[EXAMPLE]["files"][S.GROUP_KEYS[group]].get("hidden_states")
    if hfile:                                   # simulated data only
        hidden = np.loadtxt(Path(S.DATA_INDEX).parent / hfile, delimiter=",", skiprows=1)
        t = data[group][EXAMPLE][0][:, 0]
        h = hidden[np.rint(t * fs).astype(int), 1].astype(int)[:-1]
        line += "; agreement with the hidden states = %.2f" % np.mean(states[group] == h)
    print(line)

F.category_bars(states["GTK"], "GTK " + EXAMPLE + " Time Point Distribution" + TAG, WALK, "GTK_TimePointCategoryBars")
show(WALK / "GTK_TimePointCategoryBars.png", 600)

## A5. The Temporal Mapper graph

Temporal Mapper turns the 22-dimensional velocity time series into a directed graph: moments with similar movement, and moments that follow each other in time, are joined; groups of such moments become **nodes**, and the graph's paths follow the conversation as it moves between movement patterns. Settings: k = 5 neighbours, dmap = 3, texclude = 3 samples.

Each node is coloured by the state most of its moments carry. The distance matrix shows how similar every pair of moments is.

In [ ]:
from dyadmocap.dymo_graph import build_graph_with_samples, graph_input_velocity

tm = {}
for group in S.GROUPS:
    g_simp, members, nodesize, g = build_graph_with_samples(vel[group], S.K, S.DMAP, S.TEXCLUDE)
    tm[group] = dict(g_simp=g_simp, members=members, nodesize=np.asarray(nodesize), g=g, colorvar=states[group])
    print("%s: %d nodes, %d edges, largest node %d moments"
          % (group, g_simp.number_of_nodes(), g_simp.number_of_edges(), max(nodesize)))
    F.four_color_graph(g_simp, states[group], members,
                       "%s %s  k=%d, dmap=%d, texclude=%d%s" % (group, EXAMPLE, S.K, S.DMAP, S.TEXCLUDE, TAG),
                       WALK, group + "_FourColorGraph")
    show(WALK / (group + "_FourColorGraph.png"), 600)

t_vel = np.arange(vel["GTK"].shape[0]) / fs + 1 / (2 * fs)
F.distance_matrix(graph_input_velocity(vel["GTK"])[0], t_vel, "GTK " + EXAMPLE + " Distance Matrix" + TAG,
                  WALK, "GTK_DistanceMatrix")
show(WALK / "GTK_DistanceMatrix.png", 600)

## A6. Graph and cycle metrics of this dyad

* **Number of nodes**: how many movement states the graph has.
* **Node size**: the mean number of time points per node.
* **Edge density**: the share of possible transitions present.
* **Mean path distance**: the average number of steps from a node to all others (and from all others to it), for the whole graph and for the nodes of each state.
* **Cycles**: loops back to the same node: how many, their mean length, and how many states a loop passes through (Shannon entropy of its states; 0 = one state only).

In [ ]:
from dyadmocap.dymo_graph_metrics import whole_graph_metrics, node_labels
from dyadmocap.dymo_cycles import find_cycles, cycle_data, whole_cycle_metrics
from dyadmocap.dymo_node_distance import compute_dsimp
from dyadmocap.dymo_hubness import node_closeness_by_category

STATE_NAMES = ["Both Inactive", "Both Active", "Person 1 Active", "Person 2 Active"]

def graph_cycle_metrics(T):
    """The main graph and cycle metrics of one Temporal Mapper graph."""
    W = whole_graph_metrics(T["g_simp"], T["nodesize"])
    labels = node_labels(T["members"], T["colorvar"])
    C = whole_cycle_metrics(cycle_data(find_cycles(T["g_simp"]), labels), T["g_simp"].number_of_nodes())
    D = compute_dsimp(T["g"], T["members"])
    mpd_all = node_closeness_by_category(D, np.ones(len(labels), dtype=int), 1)[0][0]
    mpd_state = node_closeness_by_category(D, labels)[0]
    row = {"Number of nodes": W["numNodes"],
           "Node size (mean time points)": W["nodesizes"],
           "Edge density": W["density"],
           "Mean path distance": mpd_all}
    row.update({"Mean path distance, " + n: mpd_state[i] for i, n in enumerate(STATE_NAMES)})
    row.update({"Cycle count": C["CycleCount"],
                "Cycle mean length": C["CycleLenMean"],
                "Cycle entropy (mean)": C["meanEntropy"]})
    return row

# whole numbers as they are, others to 4 significant figures
pd.set_option("display.float_format", lambda x: "%d" % x if float(x).is_integer() else "%.4g" % x)

In [ ]:
table = pd.DataFrame({g: graph_cycle_metrics(tm[g]) for g in S.GROUPS})
table = table.rename(columns={"GTK": "Casual chat (GTK)", "MAN": "Conflict (MAN)"})
table.to_csv(WALK / ("OneDyad_GraphCycleMetrics_%s.csv" % EXAMPLE))
table

## Where everything is

The figures and the table are in the `walkthrough` folder of the results folder printed in the setup cell.

*Before committing this notebook, clear all outputs (Kernel → Restart Kernel and Clear Outputs of All Cells).*